In [ ]:
# In[1]:

import json

# Define the task directory
task_directory = 'rca100/cases/t035'

# Load the task.json file
with open(f'{task_directory}/task.json', 'r') as file:
    task_data = json.load(file)

# Extract alert context
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')

alert_title, alert_window, alert_entity

```
Out[1]:
```
The alert is titled "inventory接口流量下跌告警" (inventory interface traffic drop alert). It occurred between 2026-03-28 22:49:09 and 2026-03-28 22:55:53 (UTC+8). The entity that triggered the alert is an APM operation named "inventory::/api/v1/inventory/{productId}".

The original code execution output of IPython Kernel is also provided below for reference:

('inventory接口流量下跌告警', {'start': '2026-03-28T22:49:09.847333+08:00', 'end': '2026-03-28T22:55:53+08:00'}, {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'})```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file)

# List all available KPIs and their associated components
kpi_components = metrics_data[['metric', 'entity_name']].drop_duplicates()

kpi_components

```
Out[2]:
```
The `metrics.parquet` file contains various KPIs (metrics) associated with different components. Examples include `node_ready_status`, `node_memory_usage_rate`, `error_rate`, and `arms_exception_requests_error_count_raw`. These KPIs are linked to components such as nodes, frontend services, checkout, and inventory. The dataset includes a total of 292 unique metric-component pairs.

The original code execution output of IPython Kernel is also provided below for reference:

metric                      entity_name
0                            node_ready_status                                 
1932                    node_memory_usage_rate                                 
2208                     node_not_ready_status                                 
5244                      node_disk_usage_rate                                 
6210                    node_pod_running_count                                 
...                                        ...                              ...
87692                               error_rate  frontend-proxy-67d64dc4f8-4ptg6
87720                               error_rate        frontend-6df7d557f5-sgfcb
87744                               error_rate        frontend-6df7d557f5-nz7g4
87792  arms_exception_requests_error_count_raw                         checkout
87861  arms_exception_requests_error_count_raw                        inventory

[292 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Define the list of possible root cause components
root_cause_components = ['inventory', 'frontend', 'checkout', 'payment', 'cart']

# Filter metrics data for the specified components
filtered_metrics = metrics_data[metrics_data['entity_name'].str.contains('|'.join(root_cause_components), case=False, na=False)]

# Ensure 'value' column is numeric for aggregation
filtered_metrics['value'] = pd.to_numeric(filtered_metrics['value'], errors='coerce')

# Calculate global thresholds (P95) for each KPI and component
global_thresholds = filtered_metrics.groupby(['metric', 'entity_name'])['value'].quantile(0.95).reset_index()
global_thresholds.rename(columns={'value': 'p95_threshold'}, inplace=True)

global_thresholds

```
Out[3]:
```
The metrics data has been filtered to include KPIs associated with the 'inventory' service and other possible root cause components such as 'frontend', 'checkout', 'payment', and 'cart'. For each KPI and component, global thresholds (P95 values) have been calculated to aid in anomaly detection. The dataset includes 176 unique KPI-component pairs with their respective P95 thresholds.

The original code execution output of IPython Kernel is also provided below for reference:

metric     entity_name  p95_threshold
0    arms_exception_requests_error_count_raw        checkout   1.821440e+04
1    arms_exception_requests_error_count_raw       inventory   5.745000e+03
2                  arms_grouped_thread_count       inventory   1.682635e+00
3               arms_grouped_thread_cpu_time       inventory   1.685200e+10
4         arms_grouped_thread_runnable_count       inventory   2.170659e-01
..                                       ...             ...            ...
171                                 workload        checkout   6.099050e+03
172                                 workload        frontend   5.371835e+04
173                                 workload  frontend-proxy   2.693955e+04
174                                 workload       inventory   6.277700e+03
175                                 workload         payment   6.054050e+03

[176 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pytz
from datetime import datetime

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat('2026-03-28T22:49:09.847333+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-03-28T22:55:53+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert the 'time' column to numeric and filter metrics data within the alert time window
filtered_metrics['time'] = pd.to_numeric(filtered_metrics['time'], errors='coerce')
alert_start_us = int(alert_start.timestamp() * 1_000_000)  # Convert to microseconds
alert_end_us = int(alert_end.timestamp() * 1_000_000)      # Convert to microseconds
alert_window_metrics = filtered_metrics[
    (filtered_metrics['time'] >= alert_start_us) & (filtered_metrics['time'] <= alert_end_us)
]

# Merge the alert window metrics with global thresholds to identify anomalies
merged_data = alert_window_metrics.merge(global_thresholds, on=['metric', 'entity_name'], how='left')
merged_data['is_anomaly'] = merged_data['value'] > merged_data['p95_threshold']

# Filter anomalies
anomalies = merged_data[merged_data['is_anomaly']]

anomalies

```
Out[4]:
```
Within the alert time window (2026-03-28 22:49:09 to 2026-03-28 22:55:53 UTC+8), 191 anomalies were detected across various KPIs and components. These anomalies were identified by comparing the KPI values against their respective P95 thresholds. Components such as `frontend`, `frontend-proxy`, `cart`, and `inventory` exhibited anomalous behavior, indicating potential issues during this period.

The original code execution output of IPython Kernel is also provided below for reference:

time domain            entity_set                         entity_id                         entity_name  ...        value                            metric_set_id    service p95_threshold  is_anomaly
4     1774709499000000    apm         apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc             frontend-proxy::ingress  ...  6140.000000  apm@metric_set@apm.metric.apm.operation       None   6008.000000        True
16    1774709499000000    apm         apm.operation  9e69280c47b4c2f35576402946e3b714            frontend::POST /api/cart  ...  6140.000000  apm@metric_set@apm.metric.apm.operation       None   6113.600000        True
40    1774709469000000    apm         apm.operation  fb364e80b42f16618b6de13f3579b6ab  frontend::GET /api/recommendations  ...   972.000000  apm@metric_set@apm.metric.apm.operation       None    950.450000        True
41    1774709499000000    apm         apm.operation  fb364e80b42f16618b6de13f3579b6ab  frontend::GET /api/recommendations  ...   955.000000  apm@metric_set@apm.metric.apm.operation       None    950.450000        True
43    1774709559000000    apm         apm.operation  fb364e80b42f16618b6de13f3579b6ab  frontend::GET /api/recommendations  ...   962.000000  apm@metric_set@apm.metric.apm.operation       None    950.450000        True
...                ...    ...                   ...                               ...                                 ...  ...          ...                                      ...        ...           ...         ...
1934  1774709469000000    apm          apm.instance  edd1b3a198ca77c10c1e4414044e4671               cart-67f957cc88-njsn7  ...     0.900875   apm@metric_set@apm.metric.apm.instance       None      0.899647        True
1935  1774709499000000    apm          apm.instance  edd1b3a198ca77c10c1e4414044e4671               cart-67f957cc88-njsn7  ...     0.910135   apm@metric_set@apm.metric.apm.instance       None      0.899647        True
1958  1774709469000000    apm          apm.instance  009ec66a350bc3e5210aea3dc6e6bdfd           frontend-6df7d557f5-sgfcb  ...     0.213883   apm@metric_set@apm.metric.apm.instance       None      0.213761        True
1959  1774709499000000    apm          apm.instance  009ec66a350bc3e5210aea3dc6e6bdfd           frontend-6df7d557f5-sgfcb  ...     0.216833   apm@metric_set@apm.metric.apm.instance       None      0.213761        True
1998  1774709469000000    apm  apm.metric.exception  ed13cb14265f98d9eb3b89b664760adb                           inventory  ...  5964.000000      apm@metric_set@apm.metric.exception  inventory   5745.000000        True

[191 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_file)

# Convert 'startTime' and 'endTime' columns to numeric for filtering
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')

# Filter traces within the alert time window
alert_start_ns = alert_start_us * 1_000  # Convert microseconds to nanoseconds
alert_end_ns = alert_end_us * 1_000      # Convert microseconds to nanoseconds
alert_window_traces = traces_data[
    (traces_data['startTime'] >= alert_start_ns) & (traces_data['endTime'] <= alert_end_ns)
]

# Filter traces involving the 'inventory' service and other faulty components
faulty_components = ['inventory', 'frontend', 'frontend-proxy', 'cart']
faulty_traces = alert_window_traces[
    alert_window_traces['serviceName'].str.contains('|'.join(faulty_components), case=False, na=False)
]

# Identify the most downstream faulty service in the call chain
downstream_faulty_traces = faulty_traces.sort_values(by='startTime', ascending=False)

downstream_faulty_traces

```
Out[5]:
```
The filtered traces within the alert time window include interactions involving the 'inventory', 'frontend', 'frontend-proxy', and 'cart' services. By sorting the traces by start time in descending order, the most downstream faulty service in the call chain can be identified. However, due to the large size of the dataset (112,766 rows), further analysis is required to pinpoint the exact downstream service responsible for the issue.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId kind                              spanName  ...  __topic__      __tag__:__pack_id__                                             events links    ip
499998  7e42ebc50ac75c426f0718ab5e0879f9  7b0c5225b7d725a1  604327fb6e4a3916    3                router frontend egress  ...             5D90A72A5FB16F39-254E07                                               None  None  None
499999  7e42ebc50ac75c426f0718ab5e0879f9  604327fb6e4a3916  b617528a8069f55d    2                               ingress  ...             5D90A72A5FB16F39-254E07                                               None  None  None
499987  6a9be0d310eac08f1ce280d38af83129  7064c7342edd7503  756ca8348a31fb9f    3                router frontend egress  ...             5D90A72A5FB16F39-254E07                                               None  None  None
499988  6a9be0d310eac08f1ce280d38af83129  756ca8348a31fb9f                      2                               ingress  ...             5D90A72A5FB16F39-254E07                                               None  None  None
499982  bfc537d4a243da5c45b7295013d1cffd  966db0f1ad0731a8  ded138f21ed98e00    3                router frontend egress  ...             5D90A72A5FB16F39-254E07                                               None  None  None
...                                  ...               ...               ...  ...                                   ...  ...        ...                      ...                                                ...   ...   ...
301319  03669187701d5c36a5bdb5260731b0d2  8f53d0fc07d070b0  2c23b7ecdd7bd698    2         GET /api/products/{productId}  ...             6703BFF3ADFED5A4-2526E8                                               None  None  None
301429  03669187701d5c36a5bdb5260731b0d2  2c23b7ecdd7bd698  e2d9a0ba83c0cb84    2                                   GET  ...             6703BFF3ADFED5A4-2526E8                                               None  None  None
300282  03669187701d5c36a5bdb5260731b0d2  e2d9a0ba83c0cb84  a5b26e64f833c6bd    3                router frontend egress  ...             6703BFF3ADFED5A4-2526E6                                               None  None  None
300283  03669187701d5c36a5bdb5260731b0d2  a5b26e64f833c6bd                      2                               ingress  ...             6703BFF3ADFED5A4-2526E6                                               None  None  None
302006  4ef40b15c4778611a0ba98ad8498ba1e  c38a0b4c5079714f  8fe3781b924fe56e    2  POST /oteldemo.CartService/EmptyCart  ...             6703BFF3ADFED5A4-2526E8  [{"attributes":{},"name":"Empty cart","timesta...  None  None

[112766 rows x 21 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Filter traces where the 'inventory' service is involved
inventory_traces = faulty_traces[faulty_traces['serviceName'].str.contains('inventory', case=False, na=False)]

# Check if 'inventory' is the last faulty service in the call chain
# A service is considered downstream if it has no child spans (i.e., its spanId is not a parentSpanId in the dataset)
downstream_check = inventory_traces[~inventory_traces['spanId'].isin(faulty_traces['parentSpanId'])]

# If 'inventory' is not the last, identify the downstream faulty service
if not downstream_check.empty:
    most_downstream_service = downstream_check[['traceId', 'spanId', 'serviceName', 'spanName', 'startTime', 'endTime']]
else:
    # Identify the downstream faulty service causing the issue
    downstream_faulty_service = faulty_traces[faulty_traces['parentSpanId'].isin(inventory_traces['spanId'])]
    most_downstream_service = downstream_faulty_service[['traceId', 'spanId', 'serviceName', 'spanName', 'startTime', 'endTime']]

most_downstream_service

```
Out[6]:
```
The analysis of the filtered traces shows that the 'inventory' service is the most downstream faulty service in the call chain. The traces indicate that the 'inventory' service is involved in database operations such as `SELECT inventory.inventory` and connection handling (`HikariDataSource.getConnection`). No further downstream faulty services were identified beyond 'inventory'. This suggests that the root cause of the issue lies within the 'inventory' service itself.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId serviceName                        spanName            startTime              endTime
307686  b53d3ef31a319aa4a4c44ca5c23982e9  6fe932de1a5d6b33   inventory      SELECT inventory.inventory  1774709349938005180  1774709349938477630
307699  7e5d4713231d777a4e74ac879297c376  968b2d35ae2840e0   inventory      SELECT inventory.inventory  1774709349983492944  1774709349983858826
307734  d2e4bf2cd1003efa5b4dea9b8ea4bf21  5b3445aebeb12ea5   inventory      SELECT inventory.inventory  1774709349882702621  1774709349883133397
307753  e8e284e2d87d5d268d2ceadd02899736  e029405f0c35aa67   inventory      SELECT inventory.inventory  1774709349963485990  1774709349963874893
307773  7e5d4713231d777a4e74ac879297c376  24cb40d4cec2a7b9   inventory  HikariDataSource.getConnection  1774709349983065264  1774709349983079655
...                                  ...               ...         ...                             ...                  ...                  ...
493489  2f4c69efd3ab0aba2340e96398c80608  127646484b771d6c   inventory      SELECT inventory.inventory  1774709670947445552  1774709670947834214
493491  6f6f46a63eb42e432d373eedbd18bede  fc1d77229ea4e61f   inventory      SELECT inventory.inventory  1774709670880141668  1774709670880527404
493493  9269c3da4c1791ba5cc0a0644e7378a7  1aed544cebf66406   inventory      SELECT inventory.inventory  1774709670805300783  1774709670805746525
493495  d88124c93435c3deac31095e982dd507  f8583c2cc1c5316c   inventory      SELECT inventory.inventory  1774709670810428126  1774709670810783920
493496  b88508bad1c25abca3d715b875bbc08e  5e41a844796ccc33   inventory      SELECT inventory.inventory  1774709670814105910  1774709670814499154

[3628 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

# Load the logs.parquet file
logs_file = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_file)

# Convert the '_time_' column to numeric for filtering
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs for the 'inventory' service within the alert time window
inventory_logs = logs_data[
    (logs_data['_time_'] >= alert_start_us) & 
    (logs_data['_time_'] <= alert_end_us) & 
    (logs_data['_container_name_'].str.contains('inventory', case=False, na=False))
]

# Search for error messages or relevant information
error_logs = inventory_logs[inventory_logs['content'].str.contains('error|exception|fail', case=False, na=False)]

error_logs

```
Out[7]:
```
No error messages or relevant information indicating the root cause were found in the logs for the 'inventory' service within the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```